# Lesson 2: Your first benchmark

**You will learn:** what a *benchmark*, a *metric* and a *score* are, and how Brain-Score checks that a subject and a benchmark fit together.
**Requires:** CPU only, no downloads, under a minute. Uses the subject from [Lesson 1](01_your_first_subject.ipynb).

A **benchmark** is a fixed test. It defines:

1. which inputs to present (the stimuli),
2. which outputs it needs from the subject (the requested channels),
3. how to compare those outputs with measured data (the **metric**), producing a **score**.

## The subject

The word-length reader from Lesson 1.

In [1]:
import numpy as np
from brainscore_core import Subject
from brainscore_core.streaming import BufferedSession, StreamEvent

class WordLengthReader(Subject):
    identifier = 'word-length-reader'
    in_channels = {'text'}
    out_channels = {'behavior'}

    def participate(self, session):
        while (event := session.next_input()) is not None:
            reading_time_ms = 200 + 30 * len(event.payload)       # longer words take longer
            session.respond(StreamEvent('behavior', reading_time_ms, t_ms=event.t_ms))

## The experiment

The benchmark is the experiment. The measured data below are made up for this lesson; in a real benchmark they would be human reading times loaded from a dataset.

A benchmark declares the channels it provides and requests. `__call__` runs one session with the subject and returns a `Score`. Here the metric is the Pearson correlation between predicted and measured reading times.

In [2]:
from brainscore_core import Benchmark, Score

class ToyReadingTimes(Benchmark):
    identifier = 'toy-reading-times'
    uses_session = True
    required_input_channels = {'text'}
    requested_output_channels = {'behavior'}

    words = ['cat', 'elephant', 'it', 'photosynthesis', 'table']
    measured_ms = np.array([260., 455., 250., 610., 340.])  # illustrative values

    def __call__(self, subject):
        inputs = [StreamEvent('text', w, t_ms=i * 500) for i, w in enumerate(self.words)]
        session = BufferedSession(inputs=inputs)
        subject.participate(session)
        predicted = np.array([event.payload for event in session.collect()], dtype=float)
        correlation = np.corrcoef(predicted, self.measured_ms)[0, 1]   # [0, 1]: the off-diagonal entry
        return Score(correlation)

### Compatibility

Before running anything, Brain-Score compares what the benchmark provides and requests with what the subject declares:

- every requested output channel must be in the subject's `out_channels`;
- at least one provided input channel must be in the subject's `in_channels`;
- every channel the subject *requires* must be provided.

If a check fails, the pair is **not applicable** (N/A). It is not run, and it is not scored as zero. Here a benchmark asks for neural activity, which the reader cannot produce.

In [3]:
from brainscore_core import CompatibilityError, check_compatibility

class ToyNeuralBenchmark(ToyReadingTimes):
    identifier = 'toy-neural'
    requested_output_channels = {'neural:IT'}

try:
    check_compatibility(WordLengthReader(), ToyNeuralBenchmark())
except CompatibilityError as error:
    print('Not applicable:', error)

Not applicable: Subject 'word-length-reader' cannot produce output channels requested by benchmark 'toy-neural': {'neural:IT'}. Subject out_channels: {'behavior'}. It has no layer mapping for region 'IT'. Use the layer-mapping explorer (brainscore_vision.tools.layer_mapping_explorer) to find and commit a region-to-layer mapping.


## Run

`brainscore.score(subject, benchmark)` first checks that the pair is compatible, then runs the benchmark. You can pass objects, as here, or registered names such as `'clip-vit-b-32'`.

In [4]:
import brainscore

result = brainscore.score(WordLengthReader(), ToyReadingTimes(), check_mem=False)
print(f'score = {float(result):.3f}')

score = 0.995


## Summary

- A **benchmark** = stimuli + requested outputs + **metric** + measured data. It returns a **score**.
- `brainscore.score` checks **compatibility** from declarations, then runs the benchmark.
- An incompatible pair is N/A, never a zero.

Registered benchmarks work the same way. For example, `brainscore.score('clip-vit-b-32', 'MajajHong2015public.IT-pls-unified')` downloads a model and a dataset and compares model activity with recordings from macaque IT cortex.

**Next:** [Lesson 3](03_wrap_a_pytorch_model.ipynb) builds a subject from a real PyTorch network instead of a hand-written rule.